# Multilingual Transliteration — Full Colab Pipeline

**Languages:** Hindi · Bengali · Tamil  
**Model:** mT5-small fine-tuned on Aksharantar  
**Optimization:** CTranslate2 INT8  
**Deployment:** HuggingFace Spaces (Gradio)

> ⚠️ **Before running:** Enable GPU — Runtime → Change runtime type → T4 GPU

## Step 1 — Mount Drive & Clone Repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [3]:
import os

REPO_URL = 'https://github.com/Amresh9811/files.git'
WORK_DIR = '/content/transliteration'

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    print('Repo already cloned. Pulling latest ...')
    !git -C {WORK_DIR} pull

%cd {WORK_DIR}
print('Working directory:', os.getcwd())

Cloning into '/content/transliteration'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 34 (delta 15), reused 28 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 39.11 KiB | 19.55 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/transliteration
Working directory: /content/transliteration


## Step 2 — Install Dependencies

In [4]:
!pip install -q -r requirements.txt
print('Installation complete.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.5 MB/s eta 0:00:00
Installation complete.


## Step 3 — Credentials & GPU Check

In [6]:
import os

# Paste token directly OR use Colab Secrets (Secrets tab on left sidebar)
# Recommended: Secrets → add key 'HF_TOKEN' → enable notebook access
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = 'YOUR_HF_TOKEN'  # fallback

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('HuggingFace login OK.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace login OK.


In [7]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Step 4 — Data Budget

Using **~35% of Aksharantar** per language to keep training time reasonable on Colab.  
Adjust `TRAIN_PER_LANG` below if you want more or less data.

In [8]:
import sys
sys.path.insert(0, '/content/transliteration')

# ── DATA BUDGET (edit here) ────────────────────────────────────────────────
TRAIN_PER_LANG = 10_500   # 35% of 50K full dataset
VAL_PER_LANG   =  1_050   # 35% of  5K
TEST_PER_LANG  =    420   # 35% of  2K
# ──────────────────────────────────────────────────────────────────────────

# Patch config at runtime so all scripts use these values
import config as _cfg
_cfg.training_config.train_samples_per_lang = TRAIN_PER_LANG
_cfg.training_config.val_samples_per_lang   = VAL_PER_LANG
_cfg.training_config.test_samples_per_lang  = TEST_PER_LANG

TOTAL_TRAIN = TRAIN_PER_LANG * len(_cfg.LANGUAGES)
TOTAL_VAL   = VAL_PER_LANG   * len(_cfg.LANGUAGES)
print(f'Data budget:')
print(f'  Train : {TRAIN_PER_LANG:,} x {len(_cfg.LANGUAGES)} langs = {TOTAL_TRAIN:,} samples')
print(f'  Val   : {VAL_PER_LANG:,} x {len(_cfg.LANGUAGES)} langs = {TOTAL_VAL:,} samples')
print(f'  Test  : {TEST_PER_LANG:,} x {len(_cfg.LANGUAGES)} langs = {TEST_PER_LANG*len(_cfg.LANGUAGES):,} samples')

Data budget:
  Train : 10,500 x 3 langs = 31,500 samples
  Val   : 1,050 x 3 langs = 3,150 samples
  Test  : 420 x 3 langs = 1,260 samples


## Step 5 — Build & Cache Dataset

In [9]:
import gc, logging
from pathlib import Path
from datasets import load_from_disk
from transformers import AutoTokenizer
from config import LANG_TOKEN, model_config
from dataset import build_multilingual_dataset, tokenise_dataset

logging.basicConfig(format='%(asctime)s | %(levelname)s | %(message)s',
                    datefmt='%H:%M:%S', level=logging.INFO)

RAW_DIR       = 'data/processed'
TOKENISED_DIR = 'data/processed/tokenised'

# Tokeniser
tokeniser = AutoTokenizer.from_pretrained(model_config.base_model_name)
tokeniser.add_special_tokens({'additional_special_tokens': list(LANG_TOKEN.values())})
print(f'Vocab size: {len(tokeniser)}')

# Dataset
if Path(TOKENISED_DIR).exists():
    print('Loading cached tokenised dataset ...')
    tokenised_ds = load_from_disk(TOKENISED_DIR)
else:
    print('Building dataset (one language at a time to save RAM) ...')
    raw_ds = build_multilingual_dataset(save_dir=RAW_DIR)
    tokenised_ds = tokenise_dataset(raw_ds, tokeniser)
    tokenised_ds.save_to_disk(TOKENISED_DIR)
    del raw_ds
    gc.collect()
    print(f'Saved to {TOKENISED_DIR}')

print('\nDataset sizes:')
for split, ds in tokenised_ds.items():
    print(f'  {split:12s}: {len(ds):>7,}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

Tokenising:   0%|          | 0/1260 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/31500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3150 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1260 [00:00<?, ? examples/s]

Saved to data/processed/tokenised

Dataset sizes:
  train       :  31,500
  validation  :   3,150
  test        :   1,260


## Step 6 — Train mT5-small

In [10]:
# ── Sanity check: verify labels are NOT all -100 before training ──────────
# Run this BEFORE training. If you see "ALL MASKED" or empty targets, stop.
from datasets import load_from_disk
from transformers import AutoTokenizer

tok_check = AutoTokenizer.from_pretrained('google/mt5-small')
tok_check.add_special_tokens({'additional_special_tokens': ['__hi__', '__bn__', '__ta__']})

ds_check = load_from_disk('data/processed/tokenised')
print(f"Train samples : {len(ds_check['train']):,}")
print(f"Val   samples : {len(ds_check['validation']):,}\n")

for i in range(3):
    sample       = ds_check['train'][i]
    ids          = sample['input_ids']
    labels       = sample['labels']
    valid_labels = [l for l in labels if l != -100]
    decoded_in   = tok_check.decode(ids, skip_special_tokens=False)
    decoded_out  = tok_check.decode(valid_labels, skip_special_tokens=True) if valid_labels else '⚠ ALL MASKED — PROBLEM!'
    print(f"[{i}] input  : {decoded_in[:80]}")
    print(f"[{i}] target : {decoded_out}")
    print(f"     label ids (first 8): {labels[:8]}")
    print()

# Quick aggregate check
import numpy as np
all_labels = ds_check['train']['labels']
masked_frac = np.mean([all(l == -100 for l in row) for row in all_labels[:500]])
print(f"Fraction of fully-masked rows (first 500): {masked_frac:.2%}  (should be 0.00%)")

Train samples : 31,500
Val   samples : 3,150

[0] input  : __ta__ manmathach</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad
[0] target : மன்மதச்
     label ids (first 8): [5384, 197528, 10464, 8700, 1, -100, -100, -100]

[1] input  : __bn__ segmanter</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
[1] target : সেগম্যান্টের
     label ids (first 8): [4230, 6288, 44422, 180519, 1, -100, -100, -100]

[2] input  : __ta__ adimarangalellaam</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><p
[2] target : அடிமரங்களெல்லாம்
     label ids (first 8): [58517, 115926, 56002, 80667, 1, -100, -100, -100]

Fraction of fully-masked rows (first 500): 0.00%  (should be 0.00%)


In [14]:
import os
# Helps PyTorch reuse fragmented VRAM blocks — prevents OOM from allocator fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# NOTE: fp16 is intentionally disabled for mT5.
# mT5's lang-token embeddings overflow immediately in fp16 → NaN loss.
# fp32 fits fine on T4 (15 GB VRAM).
#
# eval_batch_size is 8: fp32 logits are 8×64×250K×4B = 0.5 GB — safe.
# With batch_size=64 it was 3.82 GB → OOM during eval.
#
# If training was interrupted, pass --resume_from checkpoints/checkpoint-<N>
# to pick up from the last saved checkpoint.

RESUME = ''   # e.g. 'checkpoints/checkpoint-500'  — leave empty to start fresh

resume_flag = f'--resume_from {RESUME}' if RESUME else ''

!python train.py \
    --epochs 10 \
    --batch_size 32 \
    --lr 3e-4 \
    --data_dir data/processed \
    {resume_flag}

print('\nTraining complete!')

Traceback (most recent call last):
  File "/content/transliteration/train.py", line 24, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2165, in <module>
    for __name in dir(_C._VariableFunctions):
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
^C

Training complete!


In [12]:
import json
with open('results/test_results.json') as f:
    results = json.load(f)
print('=== TEST RESULTS ===')
print(json.dumps(results, indent=2, ensure_ascii=False))

=== TEST RESULTS ===
{
  "overall": {
    "overall_cer": 0.4793,
    "overall_accuracy": 0.0929
  },
  "per_language": {
    "hi": {
      "cer": 0.4868,
      "wer": 0.9119,
      "accuracy": 0.0881,
      "n_samples": 420
    },
    "bn": {
      "cer": 0.5545,
      "wer": 0.919,
      "accuracy": 0.081,
      "n_samples": 420
    },
    "ta": {
      "cer": 0.4123,
      "wer": 0.8905,
      "accuracy": 0.1095,
      "n_samples": 420
    }
  }
}


## Step 7 — Free Memory & Verify Drive Save

In [15]:
import gc, torch, os

if 'tokenised_ds' in dir():
    del tokenised_ds
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU memory: {used:.2f} / {total:.1f} GB')

DRIVE_MODEL = '/content/drive/MyDrive/transliteration_model/best_model'
if os.path.isdir(DRIVE_MODEL):
    print(f'Drive save OK  →  {DRIVE_MODEL}')
    print('Files:', os.listdir(DRIVE_MODEL))
else:
    print('Drive save not found — model is in checkpoints/best/')

GPU memory: 0.00 / 15.6 GB
Drive save OK  →  /content/drive/MyDrive/transliteration_model/best_model
Files: ['tokenizer.json', 'training_args.bin', 'tokenizer_config.json', 'model.safetensors', 'generation_config.json', 'config.json']


## Step 8 — CTranslate2 Conversion & Benchmark

In [19]:
!python convert_to_ct2.py \
    --hf_model_dir  checkpoints/best \
    --ct2_model_dir ct2_model \
    --quantization  int8 \
    --data_dir      data/processed/tokenised

print('\nConversion + benchmark complete!')

2026-03-09 19:27:08 | INFO | Converting mT5 checkpoints/best → CTranslate2 (quantization=int8)
2026-03-09 19:27:08 | INFO | Running: ct2-transformers-converter --model checkpoints/best --output_dir ct2_model --quantization int8 --force
2026-03-09 19:27:26 | INFO | Conversion complete → ct2_model
2026-03-09 19:27:26 | INFO | 
2026-03-09 19:27:26 | INFO | Loading test data for benchmarking …
2026-03-09 19:27:26 | INFO | ============================================================
2026-03-09 19:27:26 | INFO | Benchmarking HuggingFace model …
2026-03-09 19:27:26 | INFO | Loading HF model from checkpoints/best
Loading weights: 100% 192/192 [00:00<00:00, 1346.95it/s, Materializing param=shared.weight]
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and 

In [17]:
import json
with open('results/benchmark.json') as f:
    bench = json.load(f)

sizes  = bench['model_sizes']
qual   = bench['quality']
speeds = bench['speedup_ratios']

print('=== BENCHMARK SUMMARY ===')
print(f"Model size   HF : {sizes['hf_mb']} MB")
print(f"             CT2: {sizes['ct2_mb']} MB  ({sizes['size_reduction_pct']}% smaller)")
print(f"CER          HF : {qual['hf_cer']}")
print(f"             CT2: {qual['ct2_cer']}  (delta {qual['cer_delta']:+.4f})")
print('Speedup ratios:')
for k, v in speeds.items():
    print(f'  {k}: {v}x')

FileNotFoundError: [Errno 2] No such file or directory: 'results/benchmark.json'

In [ ]:
# Copy CT2 model to Drive, then delete local HF checkpoint
import shutil, gc, torch
from pathlib import Path

DRIVE_CT2 = '/content/drive/MyDrive/transliteration_model/ct2_model'
if Path('ct2_model').exists():
    shutil.copytree('ct2_model', DRIVE_CT2, dirs_exist_ok=True)
    print(f'CT2 model saved to Drive: {DRIVE_CT2}')

if Path('checkpoints/best').exists():
    shutil.rmtree('checkpoints/best')
    print('Deleted local checkpoints/best (already on Drive).')

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleanup done.')

## Step 9 — Upload to HuggingFace Hub

In [21]:
from huggingface_hub import HfApi, create_repo

api      = HfApi()
USERNAME = 'Avinyaa'

# Upload fine-tuned HF model
HF_MODEL_REPO = f'{USERNAME}/multilingual-transliteration-mt51'
DRIVE_BEST    = '/content/drive/MyDrive/transliteration_model/best_model'

create_repo(HF_MODEL_REPO, repo_type='model', exist_ok=True)
print(f'Uploading HF model ...')
api.upload_folder(folder_path=DRIVE_BEST, repo_id=HF_MODEL_REPO, repo_type='model')
print(f'HF model  →  https://huggingface.co/{HF_MODEL_REPO}')

Uploading HF model ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...best_model/tokenizer.json: 100%|#########9| 16.0MB / 16.0MB            

  ...t_model/model.safetensors:   0%|          | 11.6kB / 2.23GB            

  ...t_model/training_args.bin:  30%|##9       | 1.59kB / 5.33kB            

HF model  →  https://huggingface.co/Avinyaa/multilingual-transliteration-mt51


In [22]:
import os

# Upload CT2 model
CT2_REPO  = f'{USERNAME}/multilingual-transliteration-ct2'
LOCAL_CT2 = 'ct2_model'

create_repo(CT2_REPO, repo_type='model', exist_ok=True)
print('Uploading CT2 model ...')
api.upload_folder(folder_path=LOCAL_CT2, repo_id=CT2_REPO, repo_type='model')
print(f'CT2 model  →  https://huggingface.co/{CT2_REPO}')

# Upload tokeniser files to CT2 repo so the demo can load them
for fname in ['tokenizer.json', 'tokenizer_config.json',
              'special_tokens_map.json', 'spiece.model']:
    fpath = f'{DRIVE_BEST}/{fname}'
    if os.path.exists(fpath):
        api.upload_file(path_or_fileobj=fpath, path_in_repo=fname,
                        repo_id=CT2_REPO, repo_type='model')
print('Tokeniser uploaded to CT2 repo.')

Uploading CT2 model ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ation/ct2_model/model.bin:   1%|          | 3.67MB /  432MB            

CT2 model  →  https://huggingface.co/Avinyaa/multilingual-transliteration-ct2


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...best_model/tokenizer.json: 100%|##########| 16.0MB / 16.0MB            

Tokeniser uploaded to CT2 repo.


In [23]:
# Delete local CT2 — safely on Drive + Hub now
import shutil, gc
if shutil.os.path.exists('ct2_model'):
    shutil.rmtree('ct2_model')
    print('Deleted local ct2_model.')
gc.collect()

Deleted local ct2_model.


684

## Step 10 — Deploy Gradio Demo to HuggingFace Spaces

In [26]:
from huggingface_hub import HfApi, create_repo
import os

api         = HfApi()
USERNAME    = 'Avinyaa'
SPACES_REPO = f'{USERNAME}/multilingual-transliteration'

create_repo(SPACES_REPO, repo_type='space', space_sdk='gradio', exist_ok=True)

# Patch model repo IDs in app.py before uploading
with open('app.py') as f:
    src = f.read()
src = src.replace('avinyaa/multilingual-transliteration-mt51',
                  f'{USERNAME}/multilingual-transliteration-mt51')
src = src.replace('avinyaa/multilingual-transliteration-ct2',
                  f'{USERNAME}/multilingual-transliteration-ct2')
with open('app.py', 'w') as f:
    f.write(src)

for fname in ['app.py', 'config.py', 'evaluate.py', 'requirements.txt']:
    if os.path.exists(fname):
        api.upload_file(path_or_fileobj=fname, path_in_repo=fname,
                        repo_id=SPACES_REPO, repo_type='space')
        print(f'  Uploaded: {fname}')

# Upload Spaces README (provides the gradio metadata header)
if os.path.exists('spaces_README.md'):
    api.upload_file(path_or_fileobj='spaces_README.md', path_in_repo='README.md',
                    repo_id=SPACES_REPO, repo_type='space')
    print('  Uploaded: README.md')

print(f'\nSpace live  →  https://huggingface.co/spaces/{SPACES_REPO}')

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  Uploaded: app.py
  Uploaded: config.py


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  Uploaded: evaluate.py
  Uploaded: requirements.txt


No files have been modified since last commit. Skipping to prevent empty commit.


  Uploaded: README.md

Space live  →  https://huggingface.co/spaces/Avinyaa/multilingual-transliteration


In [27]:
# Set environment variables on the Space
from huggingface_hub import HfApi

api         = HfApi()
USERNAME    = 'avinyaa'
SPACES_REPO = f'{USERNAME}/multilingual-transliteration'

api.add_space_variable(SPACES_REPO, 'HF_MODEL_ID',
                       f'{USERNAME}/multilingual-transliteration-mt51')
api.add_space_variable(SPACES_REPO, 'HF_CT2_ID',
                       f'{USERNAME}/multilingual-transliteration-ct2')
print('Space env vars set.')
print(f'Live demo  →  https://huggingface.co/spaces/{SPACES_REPO}')

Space env vars set.
Live demo  →  https://huggingface.co/spaces/avinyaa/multilingual-transliteration


## Step 11 — Smoke Test

In [28]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_REPO  = 'avinyaa/multilingual-transliteration-mt51'
LANG_TOKENS = {'Hindi': '__hi__', 'Bengali': '__bn__', 'Tamil': '__ta__'}
TEST_WORDS  = ['namaste', 'pyar', 'kitab', 'dilli']

tok   = AutoTokenizer.from_pretrained(MODEL_REPO)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPO).eval()

print(f'{"Word":<12}', end='')
for lang in LANG_TOKENS: print(f'{lang:<18}', end='')
print('\n' + '-' * 66)

for word in TEST_WORDS:
    print(f'{word:<12}', end='')
    for lang, token in LANG_TOKENS.items():
        inp = tok(f'{token} {word}', return_tensors='pt')
        with torch.no_grad():
            out = model.generate(**inp, num_beams=4, max_length=64)
        print(f'{tok.decode(out[0], skip_special_tokens=True):<18}', end='')
    print()

config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.0M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

Word        Hindi             Bengali           Tamil             
------------------------------------------------------------------
namaste     नासाती            নসাদে             நாசமே             
pyar        पीर               পৌর               பியர்             
kitab       पुस्तक            কথাক              புத்தகம்          
dilli       दिल्ली            ভুলি              டொல்ட்            


---
## Summary

| Step | Output |
|------|--------|
| Training | `checkpoints/best/` → Drive: `transliteration_model/best_model/` |
| Benchmark | `results/benchmark.json` |
| HF Model | `avinyaa/multilingual-transliteration-mt51` |
| CT2 Model | `avinyaa/multilingual-transliteration-ct2` |
| Demo | `https://huggingface.co/spaces/avinyaa/multilingual-transliteration` |

**Data used:** 17,500 train + 1,750 val + 700 test per language (35% of full dataset)